<a href="https://colab.research.google.com/github/Tulja10/IPL_Data_Analysis_Project/blob/main/IPL_DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import kagglehub
path = kagglehub.dataset_download("patrickb1912/ipl-complete-dataset-20082020")
print(path)

Using Colab cache for faster access to the 'ipl-complete-dataset-20082020' dataset.
/kaggle/input/ipl-complete-dataset-20082020


In [7]:
import os
os.listdir(path)

['matches.csv', 'deliveries.csv']

In [8]:
import pandas as pd
matches=pd.read_csv('/kaggle/input/ipl-complete-dataset-20082020/matches.csv')
deliveries=pd.read_csv('/kaggle/input/ipl-complete-dataset-20082020/deliveries.csv')

In [9]:
matches.shape
deliveries.shape

matches.head()


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


1. DATASET UNDERSTANDING

In [10]:
#exploring dataset
matches.shape
matches.head()
matches.tail()
# knowing columns and their dtypes
matches.info()
# mathematical summary of numerical cols
matches.describe()
matches.isnull().sum()
# How many unique seasons, teams and venues are there?
matches.nunique()

deliveries.shape
deliveries.head()
deliveries.tail()
# knowing columns and their dtypes
deliveries.info()
# mathematical summary of numerical cols
deliveries.describe()
deliveries.isnull().sum()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   object 
 2   city             1044 non-null   object 
 3   date             1095 non-null   object 
 4   match_type       1095 non-null   object 
 5   player_of_match  1090 non-null   object 
 6   venue            1095 non-null   object 
 7   team1            1095 non-null   object 
 8   team2            1095 non-null   object 
 9   toss_winner      1095 non-null   object 
 10  toss_decision    1095 non-null   object 
 11  winner           1090 non-null   object 
 12  result           1095 non-null   object 
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   object 
 17  method        

,0
match_id,0
inning,0
batting_team,0
bowling_team,0
over,0
ball,0
batter,0
bowler,0
non_striker,0
batsman_runs,0


2.DATA CLEANING

In [11]:
matches['team1'].unique() #19
# standardizing team names
def standardize(team):
  if team=='Royal Challengers Bangalore':
    return 'Royal Challengers Bengaluru'
  elif team=='Delhi Daredevils':
    return 'Delhi Capitals'
  elif team=='Deccan Chargers':
    return 'Sunrisers Hyderabad'
  elif team=='Kings XI Punjab':
    return 'Punjab Kings'
  elif team=='Pune Warriors':
    return 'Rising Pune Supergiants'
  elif team=='Rising Pune Supergiant':
    return 'Rising Pune Supergiants'
  elif team=='Gujarat Lions':
    return 'Gujarat Titans'
  else:
    return team

matches['team1']=matches['team1'].apply(standardize)
matches['team2']=matches['team2'].apply(standardize)
matches['winner']=matches['winner'].apply(standardize)

matches['winner'].nunique() # count of teams reduced from 19 to 12

12

2. TEAM ANALYSIS

Which team has played the most matches?

In [45]:
pd.concat([matches['team1'],matches['team2']]).value_counts().head(5)

,count
Mumbai Indians,261
Sunrisers Hyderabad,257
Royal Challengers Bengaluru,255
Delhi Capitals,252
Kolkata Knight Riders,251


Which teams have won the most matches?

In [46]:
matches['winner'].value_counts().head(5)

,count
winner,
Mumbai Indians,144
Chennai Super Kings,138
Kolkata Knight Riders,131
Royal Challengers Bengaluru,123
Sunrisers Hyderabad,117


Which teams have the highest win percentage?

In [47]:
((matches['winner'].value_counts()/(matches['team1'].value_counts()+matches['team2'].value_counts()))*100).head(5)

,count
Chennai Super Kings,57.983193
Delhi Capitals,45.634921
Gujarat Titans,54.666667
Kochi Tuskers Kerala,42.857143
Kolkata Knight Riders,52.191235


Which team have won the most titles?

In [15]:
matches[matches['match_type']=='Final']['winner'].value_counts().head(1)

,count
winner,
Mumbai Indians,5


Which teams have the best chasing record?

In [16]:
df=matches[(matches['toss_winner']==matches['winner']) & (matches['toss_decision']=='field') | (matches['toss_winner']!=matches['winner']) & (matches['toss_decision']=='bat')]
df.groupby('winner')['target_runs'].max().sort_values(ascending=False).head(1)

,target_runs
winner,
Punjab Kings,262.0


3. TOSS ANALYSIS

Does winning the toss increase chances of winning?

In [17]:
((matches['toss_winner']==matches['winner']).sum()/matches.shape[0])*100

np.float64(35.52511415525114)

How often do teams choose batting vs bowling after winning the toss?

In [18]:
(matches['toss_decision'].value_counts()/len(matches))*100

,count
toss_decision,
field,64.292237
bat,35.707763


Which teams benefit most from winning the toss?

In [48]:
matches[matches['toss_winner']==matches['winner']]['winner'].value_counts().head(5)

,count
winner,
Mumbai Indians,78
Chennai Super Kings,75
Kolkata Knight Riders,68
Rajasthan Royals,60
Sunrisers Hyderabad,38


4. BATTING ANALYSIS

Who are the top 10 run scorers in IPL history?

In [49]:
deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(5)

,batsman_runs
batter,
V Kohli,8014
S Dhawan,6769
RG Sharma,6630
DA Warner,6567
SK Raina,5536


Which batters have scored the most centuries?

In [50]:
temp=deliveries.groupby(['match_id','batter'])['batsman_runs'].sum().reset_index()
temp[temp['batsman_runs']>=100]['batter'].value_counts().head(5)

,count
batter,
V Kohli,8
JC Buttler,7
CH Gayle,6
DA Warner,4
SR Watson,4


Which batters have scored the most fifties?

In [51]:
temp[temp['batsman_runs']>=50]['batter'].value_counts().head(5)

,count
batter,
DA Warner,66
V Kohli,64
S Dhawan,53
RG Sharma,45
AB de Villiers,44


Which batters hit the most fours?

In [52]:
tempdf=deliveries[deliveries['batsman_runs']==4]
tempdf.groupby('batter')['batsman_runs'].count().sort_values(ascending=False).head(5)

,batsman_runs
batter,
S Dhawan,768
V Kohli,708
DA Warner,663
RG Sharma,599
SK Raina,506


Which batters hit the most sixes?

In [53]:
tempdf2=deliveries[deliveries['batsman_runs']==6]
tempdf2.groupby('batter')['batsman_runs'].count().sort_values(ascending=False).head(5)

,batsman_runs
batter,
CH Gayle,359
RG Sharma,281
V Kohli,273
AB de Villiers,253
MS Dhoni,252


Which batters have the highest strike rate?

In [54]:
cond1=deliveries.groupby('batter')['batsman_runs'].sum() #total runs scored by batter
cond2=deliveries.groupby('batter').size() # no of balls played by batter
((cond1/cond2)*100).sort_values(ascending=False).head(5)

,0
batter,
L Wood,300.000000
B Stanlake,250.000000
J Fraser-McGurk,220.000000
R Sai Kishore,216.666667
Umar Gul,205.263158


Which batters are most consistent across seasons?

In [55]:
dfd=matches.merge(deliveries,how='inner',left_on='id',right_on='match_id').groupby(['batter','season'])['batsman_runs'].sum().reset_index()
dfd.groupby('batter')['batsman_runs'].std().sort_values().head(5)

,batsman_runs
batter,
CRD Fernando,0.00000
RP Meredith,0.00000
KK Ahmed,0.50000
BB Sran,0.57735
A Singh,0.57735


5. BOWLING ANALYSIS

Who are the highest wicket takers?

In [57]:
deliveries.groupby('bowler')['is_wicket'].sum().sort_values(ascending=False).head(5)
# OR
# deliveries[deliveries['is_wicket']==1].groupby('bowler').size()

,is_wicket
bowler,
YS Chahal,213
DJ Bravo,207
PP Chawla,201
SP Narine,200
R Ashwin,198


Which bowlers bowl the most dot balls?

In [58]:
deliveries[deliveries['total_runs']==0].groupby('bowler').size().sort_values(ascending=False).head(5)

,0
bowler,
B Kumar,1632
SP Narine,1569
R Ashwin,1552
PP Chawla,1325
Harbhajan Singh,1263


Which bowlers concede the fewest runs based on number of balls he bowled?

In [29]:
bowler_stats=deliveries.groupby('bowler').agg({'total_runs':'sum',
                                  'ball':'count'})
bowler_stats=bowler_stats[bowler_stats['ball']>=300] #considering only regular bowlers who played bowled atleast 300 balls
bowler_stats.sort_values(['total_runs','ball'],ascending=[True,True])


,total_runs,ball
bowler,,
AD Mascarenhas,365,310
GD McGrath,366,329
ST Jayasuriya,396,301
BA Bhatt,408,303
AC Thomas,416,327
...,...,...
YS Chahal,4681,3628
RA Jadeja,4917,3895
B Kumar,5051,4060


Which bowlers have the best economy rate?

In [59]:
(bowler_stats['total_runs']/(bowler_stats['ball']/6)).sort_values().head(5)

,0
bowler,
A Kumble,6.646999
GD McGrath,6.674772
M Muralitharan,6.698292
J Yadav,6.738693
SP Narine,6.761216


Which bowlers have the best bowling strike rate?

In [60]:
(bowler_stats['ball']/deliveries[deliveries['is_wicket']==1].groupby('bowler').size()).sort_values().head(5)

,0
bowler,
L Ngidi,12.214286
M Pathirana,13.236842
MF Maharoof,13.363636
Mukesh Kumar,13.548387
DE Bollinger,13.953488


6. VENUE ANALYSIS

Which venues host the most matches?

In [32]:
#standardizing venues
def standardize_venue(venue):

    venue_mapping = {

        # Chinnaswamy
        'M.Chinnaswamy Stadium':
            'M Chinnaswamy Stadium',
        'M Chinnaswamy Stadium, Bengaluru':
            'M Chinnaswamy Stadium',

        # Arun Jaitley / Feroz Shah Kotla
        'Feroz Shah Kotla':
            'Arun Jaitley Stadium',
        'Arun Jaitley Stadium, Delhi':
            'Arun Jaitley Stadium',

        # Wankhede
        'Wankhede Stadium, Mumbai':
            'Wankhede Stadium',

        # Chepauk
        'MA Chidambaram Stadium':
            'MA Chidambaram Stadium, Chepauk',
        'MA Chidambaram Stadium, Chepauk, Chennai':
            'MA Chidambaram Stadium, Chepauk',

        # Rajiv Gandhi
        'Rajiv Gandhi International Stadium':
            'Rajiv Gandhi International Stadium, Uppal',
        'Rajiv Gandhi International Stadium, Uppal, Hyderabad':
            'Rajiv Gandhi International Stadium, Uppal',

        # Punjab
        'Punjab Cricket Association Stadium, Mohali':
            'Punjab Cricket Association IS Bindra Stadium',
        'Punjab Cricket Association IS Bindra Stadium, Mohali':
            'Punjab Cricket Association IS Bindra Stadium',
        'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh':
            'Punjab Cricket Association IS Bindra Stadium',

        # Brabourne
        'Brabourne Stadium, Mumbai':
            'Brabourne Stadium',

        # DY Patil
        'Dr DY Patil Sports Academy, Mumbai':
            'Dr DY Patil Sports Academy',

        # MCA Pune
        'Maharashtra Cricket Association Stadium, Pune':
            'Maharashtra Cricket Association Stadium',

        # Eden Gardens
        'Eden Gardens, Kolkata':
            'Eden Gardens',

        # Narendra Modi / Motera
        'Sardar Patel Stadium, Motera':
            'Narendra Modi Stadium',

        # Vizag
        'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam':
            'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',

    }

    return venue_mapping.get(venue, venue)

matches['venue'] = matches['venue'].apply(standardize_venue)
matches.venue.unique()
matches.venue.value_counts().head(5)


,count
venue,
Wankhede Stadium,118
M Chinnaswamy Stadium,94
Eden Gardens,93
Arun Jaitley Stadium,90
"MA Chidambaram Stadium, Chepauk",85


Which venues have the highest average score?

In [33]:
matches.merge(deliveries,left_on='id',right_on='match_id').groupby('venue')['total_runs'].mean().sort_values(ascending=False).head(5)

,total_runs
venue,
"Himachal Pradesh Cricket Association Stadium, Dharamsala",1.542770
Holkar Cricket Stadium,1.461578
"Sawai Mansingh Stadium, Jaipur",1.443460
"Narendra Modi Stadium, Ahmedabad",1.439929
Brabourne Stadium,1.425835


Which venues produce the most sixes?

In [34]:
deliveries[deliveries['batsman_runs']==6].merge(matches,left_on='match_id',right_on='id').groupby('venue').size().sort_values(ascending=False).head(5)

,0
venue,
Wankhede Stadium,1611
M Chinnaswamy Stadium,1402
Eden Gardens,1183
Arun Jaitley Stadium,1133
"MA Chidambaram Stadium, Chepauk",924


Which venues have the highest wicket rate?

In [42]:
no_of_matches=matches.venue.value_counts()
wicket_data=deliveries[deliveries['is_wicket']==1]
no_of_wickets=wicket_data.merge(matches,left_on='match_id',right_on='id').groupby('venue').size()
(no_of_wickets/no_of_matches).sort_values(ascending=False).head(5)

,0
venue,
"Vidarbha Cricket Association Stadium, Jamtha",16.000000
"Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur",15.800000
"Himachal Pradesh Cricket Association Stadium, Dharamsala",14.000000
Kingsmead,13.400000
Newlands,13.285714


7. SEASON ANALYSIS

Which season was the highest scoring?

In [40]:
matches.groupby('season')['target_runs'].max().sort_values(ascending=False).head(1)

,target_runs
season,
2024,288.0


Which season saw the most wickets?

In [44]:
wicket_data.merge(matches,left_on='match_id',right_on='id').groupby('season').size().sort_values(ascending=False).head(1)

,0
season,
2023,916
